# Embeddings pre-treinados: Word2Vec, FastText e GloVe

Este notebook testa modelos pre-treinados usando tambem o JSON dos artigos STIL 2023 como fonte de palavras e pares para consulta. Ele carrega vetores prontos, compara palavras por similaridade e permite visualizar palavras proximas em 2D.

Observacao: os modelos Word2Vec Google News e FastText Wiki News sao grandes e podem demorar no Colab. Se quiser um teste rapido, carregue primeiro apenas o GloVe.

In [ ]:
%pip install -q gensim scikit-learn pandas matplotlib nltk


In [ ]:
import json
import re
from collections import Counter

import gensim.downloader as api
import matplotlib.pyplot as plt
import nltk
import pandas as pd
from gensim.models import Word2Vec
from nltk.corpus import stopwords
from sklearn.decomposition import PCA

nltk.download("stopwords")

## Carregar o JSON dos artigos

Este notebook usa modelos prontos, mas as palavras consultadas saem do corpus STIL 2023. No Colab, faca upload do arquivo `stil2023_articles.json` ou `stil2023_articles (1).json`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    json_path = next(iter(uploaded.keys()))
except Exception:
    json_path = "stil2023_articles (1).json"

with open(json_path, "r", encoding="utf-8") as f:
    artigos = json.load(f)

print(f"Artigos carregados: {len(artigos)}")

In [ ]:
def reparar_mojibake(texto):
    if not isinstance(texto, str):
        return ""
    try:
        return texto.encode("latin1").decode("utf-8")
    except Exception:
        return texto


def extrair_texto_artigo(artigo):
    partes = []
    for campo in ["titulo", "resumo", "keywords", "artigo_completo"]:
        valor = artigo.get(campo, "")
        if isinstance(valor, list):
            valor = " ".join(str(item) for item in valor)
        partes.append(reparar_mojibake(str(valor)))
    return " ".join(partes).lower()


def tokenizar(texto):
    return re.findall(r"[a-zA-Z][a-zA-Z\-]{2,}", texto.lower())


tokens_corpus = []
for artigo in artigos:
    tokens_corpus.extend(tokenizar(extrair_texto_artigo(artigo)))

frequencias_corpus = Counter(tokens_corpus)
vocabulario_corpus = [palavra for palavra, freq in frequencias_corpus.most_common() if freq >= 2]

display(pd.DataFrame(frequencias_corpus.most_common(20), columns=["palavra", "frequencia"]))

## Modelos disponíveis

- `glove-wiki-gigaword-100`: GloVe pré-treinado, menor e mais rápido.
- `word2vec-google-news-300`: Word2Vec pré-treinado, grande.
- `fasttext-wiki-news-subwords-300`: FastText pré-treinado, grande e com suporte a subpalavras.

In [ ]:
MODELOS_PRE_TREINADOS = {
    "glove": "glove-wiki-gigaword-100",
    "word2vec": "word2vec-google-news-300",
    "fasttext": "fasttext-wiki-news-subwords-300",
}

# Para Colab com pouco tempo, use ["glove"].
# Para cumprir a comparação completa, use ["glove", "word2vec", "fasttext"].
modelos_para_carregar = ["word2vec"]

modelos = {}
for nome, codigo in MODELOS_PRE_TREINADOS.items():
    if nome in modelos_para_carregar:
        print(f"Carregando {nome}: {codigo}")
        modelos[nome] = api.load(codigo)
        print(f"{nome} carregado com {len(modelos[nome].index_to_key):,} palavras.")

## Atividades: Word2Vec pre-treinado x Word2Vec dos artigos

Esta secao responde a atividade usando o mesmo JSON dos artigos:

- treina um Word2Vec Skip-gram com os artigos (`sg=1`);
- identifica `modelos`, `linguagem`, o substantivo mais frequente e o verbo mais frequente;
- mostra vetores e termos similares;
- compara soma/subtracao de vetores no Word2Vec pre-treinado e no Word2Vec treinado com os artigos.

In [ ]:
stop_pt = set(stopwords.words("portuguese"))
stop_en = set(stopwords.words("english"))
stop_all = stop_pt | stop_en

sentencas_artigos = []
tokens_filtrados_artigos = []

for artigo in artigos:
    tokens = tokenizar(extrair_texto_artigo(artigo))
    tokens = [token for token in tokens if len(token) > 2]
    if tokens:
        sentencas_artigos.append(tokens)
        tokens_filtrados_artigos.extend([token for token in tokens if token not in stop_all])

frequencias_filtradas = Counter(tokens_filtrados_artigos)
display(pd.DataFrame(frequencias_filtradas.most_common(20), columns=["palavra", "frequencia"]))

In [ ]:
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 2
EPOCHS = 30

word2vec_artigos = Word2Vec(
    sentences=sentencas_artigos,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=2,
    sg=1,
    epochs=EPOCHS,
)

print("Word2Vec dos artigos treinado com Skip-gram.")
print("Vocabulario:", len(word2vec_artigos.wv))

In [ ]:
def normalizar_token_pos(token):
    token = reparar_mojibake(str(token)).lower()
    encontrados = tokenizar(token)
    return encontrados[0] if encontrados else ""


def frequencias_por_pos(pos_alvos):
    contador = Counter()
    for artigo in artigos:
        tokens = artigo.get("artigo_tokenizado", []) or []
        tags = artigo.get("pos_tagger", []) or []
        for token, tag in zip(tokens, tags):
            token_limpo = normalizar_token_pos(token)
            if not token_limpo or len(token_limpo) <= 2:
                continue
            if token_limpo in stop_all:
                continue
            if tag in pos_alvos and token_limpo in word2vec_artigos.wv:
                contador[token_limpo] += 1
    return contador


substantivos = frequencias_por_pos({"NOUN", "PROPN"})
verbos = frequencias_por_pos({"VERB", "AUX"})

substantivo_mais_frequente = substantivos.most_common(1)[0][0] if substantivos else None
verbo_mais_frequente = verbos.most_common(1)[0][0] if verbos else None

termos_atividade = {
    "modelos": "modelos",
    "linguagem": "linguagem",
    "substantivo_mais_frequente": substantivo_mais_frequente,
    "verbo_mais_frequente": verbo_mais_frequente,
}

display(pd.DataFrame(
    [{"categoria": categoria, "termo": termo} for categoria, termo in termos_atividade.items()]
))

In [ ]:
def vetor_word2vec_artigos(palavra):
    if palavra is None:
        return pd.DataFrame({"aviso": ["Termo nao encontrado no corpus."]})
    if palavra not in word2vec_artigos.wv:
        return pd.DataFrame({"aviso": [f"'{palavra}' nao esta no vocabulario do Word2Vec dos artigos."]})
    return pd.DataFrame({
        "dimensao": list(range(word2vec_artigos.vector_size)),
        "valor": word2vec_artigos.wv[palavra],
    })


for categoria, termo in termos_atividade.items():
    print(f"\nVetor gerado no Word2Vec dos artigos - {categoria}: {termo}")
    display(vetor_word2vec_artigos(termo).head(20))

In [ ]:
def similares_artigos(palavra, topn=10):
    if palavra is None:
        return pd.DataFrame({"aviso": ["Termo nao encontrado no corpus."]})
    if palavra not in word2vec_artigos.wv:
        return pd.DataFrame({"aviso": [f"'{palavra}' nao esta no vocabulario do Word2Vec dos artigos."]})
    return pd.DataFrame(
        word2vec_artigos.wv.most_similar(palavra, topn=topn),
        columns=["palavra", "similaridade"],
    )


def similares_pretreinado(palavra, topn=10):
    if not modelos:
        return pd.DataFrame({"aviso": ["Nenhum modelo pre-treinado foi carregado."]})
    nome_modelo, modelo = next(iter(modelos.items()))
    if palavra is None:
        return pd.DataFrame({"aviso": ["Termo nao encontrado no corpus."]})
    if palavra not in modelo:
        return pd.DataFrame({"aviso": [f"'{palavra}' nao esta no vocabulario do modelo pre-treinado {nome_modelo}."]})
    return pd.DataFrame(
        modelo.most_similar(palavra, topn=topn),
        columns=["palavra", "similaridade"],
    )


for categoria, termo in termos_atividade.items():
    print(f"\nTermos similares no Word2Vec dos artigos - {categoria}: {termo}")
    display(similares_artigos(termo, topn=10))

    print(f"Termos similares no Word2Vec pre-treinado - {categoria}: {termo}")
    display(similares_pretreinado(termo, topn=10))

In [ ]:
def analogia(modelo_vetores, positivos, negativos=None, topn=10, nome_modelo="modelo"):
    negativos = negativos or []
    termos = positivos + negativos
    ausentes = [termo for termo in termos if termo not in modelo_vetores]
    if ausentes:
        return pd.DataFrame({"aviso": [f"Termos fora do vocabulario em {nome_modelo}: {', '.join(ausentes)}"]})
    return pd.DataFrame(
        modelo_vetores.most_similar(positive=positivos, negative=negativos, topn=topn),
        columns=["palavra", "similaridade"],
    )


operacoes_vetoriais = [
    {
        "descricao": "modelos + linguagem - corpus",
        "positivos": ["modelos", "linguagem"],
        "negativos": ["corpus"],
    },
    {
        "descricao": "substantivo + linguagem - verbo",
        "positivos": [substantivo_mais_frequente, "linguagem"],
        "negativos": [verbo_mais_frequente],
    },
]

modelo_pretreinado_nome, modelo_pretreinado = next(iter(modelos.items())) if modelos else (None, None)

for operacao in operacoes_vetoriais:
    positivos = [termo for termo in operacao["positivos"] if termo]
    negativos = [termo for termo in operacao["negativos"] if termo]

    print(f"\nOperacao vetorial: {operacao['descricao']}")
    print("Word2Vec dos artigos")
    display(analogia(
        word2vec_artigos.wv,
        positivos=positivos,
        negativos=negativos,
        topn=10,
        nome_modelo="Word2Vec dos artigos",
    ))

    print(f"Word2Vec pre-treinado: {modelo_pretreinado_nome}")
    if modelo_pretreinado is None:
        display(pd.DataFrame({"aviso": ["Nenhum modelo pre-treinado foi carregado."]}))
    else:
        display(analogia(
            modelo_pretreinado,
            positivos=positivos,
            negativos=negativos,
            topn=10,
            nome_modelo=f"Word2Vec pre-treinado {modelo_pretreinado_nome}",
        ))

In [ ]:
def palavras_similares(modelo, palavra, topn=10):
    if palavra not in modelo:
        return pd.DataFrame({"aviso": [f"A palavra '{palavra}' nao esta no vocabulario do modelo."]})
    return pd.DataFrame(modelo.most_similar(palavra, topn=topn), columns=["palavra", "similaridade"])


def palavras_do_corpus_no_modelo(modelo, limite=30):
    palavras = [palavra for palavra in vocabulario_corpus if palavra in modelo]
    return palavras[:limite]


for nome, modelo in modelos.items():
    palavras_disponiveis = palavras_do_corpus_no_modelo(modelo, limite=30)
    palavra_teste = palavras_disponiveis[0] if palavras_disponiveis else "language"
    print(f"\nModelo: {nome}")
    print(f"Palavra escolhida do corpus: {palavra_teste}")
    display(palavras_similares(modelo, palavra_teste, topn=10))

In [ ]:
def comparar_pares(modelo, pares):
    linhas = []
    for a, b in pares:
        if a in modelo and b in modelo:
            score = modelo.similarity(a, b)
            linhas.append({"palavra_1": a, "palavra_2": b, "similaridade": score})
        else:
            linhas.append({"palavra_1": a, "palavra_2": b, "similaridade": None})
    return pd.DataFrame(linhas)


for nome, modelo in modelos.items():
    palavras_disponiveis = palavras_do_corpus_no_modelo(modelo, limite=12)
    pares = list(zip(palavras_disponiveis[::2], palavras_disponiveis[1::2]))
    if not pares:
        pares = [("language", "linguistics"), ("text", "corpus")]
    print(f"\nModelo: {nome}")
    display(comparar_pares(modelo, pares))

In [ ]:
def plotar_vizinhos(modelo, palavra, topn=20, titulo=None):
    if palavra not in modelo:
        print(f"A palavra '{palavra}' nao esta no vocabulario.")
        return

    vizinhos = [palavra] + [w for w, _ in modelo.most_similar(palavra, topn=topn)]
    vetores = [modelo[w] for w in vizinhos]
    coords = PCA(n_components=2, random_state=42).fit_transform(vetores)

    plt.figure(figsize=(10, 7))
    plt.scatter(coords[:, 0], coords[:, 1])
    for i, token in enumerate(vizinhos):
        plt.annotate(token, (coords[i, 0], coords[i, 1]))
    plt.title(titulo or f"Vizinhos de '{palavra}'")
    plt.grid(alpha=0.2)
    plt.show()


for nome, modelo in modelos.items():
    palavras_disponiveis = palavras_do_corpus_no_modelo(modelo, limite=1)
    palavra_teste = palavras_disponiveis[0] if palavras_disponiveis else "language"
    plotar_vizinhos(modelo, palavra_teste, topn=20, titulo=f"{nome}: palavras proximas de {palavra_teste}")